In [1]:
import os
import sys
import numpy as np
import re

# 1. Set Working Directory
PROJECT_DIR = '/content/drive/MyDrive/GalaxEye Space — Technical Assessment Submission/open-cd'
os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print("🚀 Initiating Phase 2: Data Pipeline Construction...")

# 2. CREATE HETERO LOADER & BINARIZER
os.makedirs('opencd/datasets/transforms', exist_ok=True)
loader_code = """import numpy as np
import rasterio
from mmcv.transforms import BaseTransform
from opencd.registry import TRANSFORMS

@TRANSFORMS.register_module()
class LoadHeteroImagesFromFile(BaseTransform):
    def transform(self, results: dict) -> dict:
        if isinstance(results['img_path'], list):
            pre_path, post_path = results['img_path'][0], results['img_path'][1]
        else:
            pre_path, post_path = results['img_path'], results['img_path2']

        with rasterio.open(pre_path) as src:
            eo = src.read().transpose(1, 2, 0).astype(np.float32)
            eo = (eo / 10000.0) if eo.max() > 255 else (eo / 255.0)
            eo = np.clip(eo, 0, 1)

        with rasterio.open(post_path) as src:
            sar = src.read().transpose(1, 2, 0).astype(np.float32)
            sar_db = 10 * np.log10(sar + 1e-8)
            s_min, s_max = np.min(sar_db), np.max(sar_db)
            sar_norm = (sar_db - s_min) / (s_max - s_min) if s_max > s_min else sar_db

        results['img'] = [eo, sar_norm]
        results['img_shape'] = eo.shape[:2]
        results['ori_shape'] = eo.shape[:2]
        return results

@TRANSFORMS.register_module()
class BinarizeLabels(BaseTransform):
    def transform(self, results: dict) -> dict:
        if 'gt_seg_map' in results:
            gt = results['gt_seg_map']
            gt = np.where(gt == 255, 1, gt)
            gt = np.where(gt > 1, 1, gt)
            results['gt_seg_map'] = gt
        return results
"""
with open('opencd/datasets/transforms/hetero_loading.py', 'w') as f: f.write(loader_code)
print("✅ Created: hetero_loading.py")


# 3. DYNAMICALLY DETECT BASE CLASS IMPORT (The Version-Proof Fix)
target_import = "from .base_cddataset import BaseCDDataset" # Fallback
base_class = "BaseCDDataset"
reference_file = 'opencd/datasets/levir_cd.py'

if os.path.exists(reference_file):
    with open(reference_file, 'r') as f:
        content = f.read()
        # Find the exact base class the current Open-CD version uses
        class_match = re.search(r'class [A-Za-z0-9_]+\((.*?)\):', content)
        if class_match:
            detected_base = class_match.group(1).strip()
            # Find the corresponding import line for that class
            for line in content.split('\n'):
                if detected_base in line and ('import ' in line or 'from ' in line):
                    target_import = line.strip()
                    base_class = detected_base
                    print(f"🔍 Dynamically matched Base Class: {base_class}")
                    print(f"🔍 Dynamically matched Import: {target_import}")
                    break

# 4. CREATE DATASET CLASS
dataset_code = f"""from opencd.registry import DATASETS
{target_import}

@DATASETS.register_module()
class DisasterCDDataset({base_class}):
    METAINFO = dict(classes=('unchanged', 'changed'), palette=[[0, 0, 0], [255, 255, 255]])
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
"""
with open('opencd/datasets/disaster_cd.py', 'w') as f: f.write(dataset_code)
print("✅ Created: disaster_cd.py")


# 5. CREATE CONFIGURATION
os.makedirs('configs/_base_/datasets', exist_ok=True)
config_code = """dataset_type = 'DisasterCDDataset'
data_root = '/content/dataset'
crop_size = (256, 256)

train_pipeline = [
    dict(type='LoadHeteroImagesFromFile'),
    dict(type='MultiImgLoadAnnotations'),
    dict(type='BinarizeLabels'),
    dict(type='MultiImgRandomRotate', prob=0.5, degree=180),
    dict(type='MultiImgRandomFlip', prob=0.5, direction='horizontal'),
    dict(type='MultiImgRandomFlip', prob=0.5, direction='vertical'),
    dict(type='MultiImgRandomCrop', crop_size=crop_size, cat_max_ratio=0.75),
    dict(type='MultiImgPackSegInputs')
]

test_pipeline = [
    dict(type='LoadHeteroImagesFromFile'),
    dict(type='MultiImgLoadAnnotations'),
    dict(type='BinarizeLabels'),
    dict(type='MultiImgPackSegInputs')
]

train_dataloader = dict(
    batch_size=8, num_workers=2, persistent_workers=True,
    sampler=dict(type='InfiniteSampler', shuffle=True),
    dataset=dict(type=dataset_type, data_root=data_root, img_suffix='.tif', seg_map_suffix='.tif',
        data_prefix=dict(img_path_from='train/pre-event', img_path_to='train/post-event', seg_map_path='train/target'),
        pipeline=train_pipeline))

val_dataloader = dict(
    batch_size=1, num_workers=2, persistent_workers=True,
    sampler=dict(type='DefaultSampler', shuffle=False),
    dataset=dict(type=dataset_type, data_root=data_root, img_suffix='.tif', seg_map_suffix='.tif',
        data_prefix=dict(img_path_from='val/pre-event', img_path_to='val/post-event', seg_map_path='val/target'),
        pipeline=test_pipeline))

test_dataloader = val_dataloader.copy()
test_dataloader['dataset']['data_prefix'] = dict(img_path_from='test/pre-event', img_path_to='test/post-event', seg_map_path='test/target')

val_evaluator = dict(type='mmseg.IoUMetric', iou_metrics=['mIoU', 'mFscore'])
test_evaluator = val_evaluator
"""
with open('configs/_base_/datasets/disaster_hetero_cd.py', 'w') as f: f.write(config_code)
print("✅ Created: disaster_hetero_cd.py")

# 6. PATCH REGISTRIES
def append_import(file_path, statement):
    if not os.path.exists(file_path): open(file_path, 'w').close()
    with open(file_path, 'r') as f: content = f.read()
    if statement not in content:
        with open(file_path, 'a') as f: f.write(f"\n{statement}\n")

append_import('opencd/datasets/__init__.py', 'from .disaster_cd import DisasterCDDataset')
append_import('opencd/datasets/transforms/__init__.py', 'from .hetero_loading import LoadHeteroImagesFromFile, BinarizeLabels')
print("✅ Patched Open-CD Registries")

# 7. WIPE CACHE & VERIFY
if 'opencd.datasets' in sys.modules: del sys.modules['opencd.datasets']
if 'opencd.datasets.transforms' in sys.modules: del sys.modules['opencd.datasets.transforms']

import opencd.datasets
import opencd.datasets.transforms
from opencd.registry import DATASETS, TRANSFORMS

print("\n--- Final Verification ---")
if 'DisasterCDDataset' in DATASETS.module_dict and 'LoadHeteroImagesFromFile' in TRANSFORMS.module_dict:
    print("🎯 SUCCESS! Phase 2 Data Pipeline is 100% operational. Proceed to Phase 3.")
else:
    print("❌ Registry failed. Please check paths.")

🚀 Initiating Phase 2: Data Pipeline Construction...
✅ Created: hetero_loading.py
🔍 Dynamically matched Base Class: _BaseCDDataset
🔍 Dynamically matched Import: from .basecddataset import _BaseCDDataset
✅ Created: disaster_cd.py
✅ Created: disaster_hetero_cd.py
✅ Patched Open-CD Registries

--- Final Verification ---
🎯 SUCCESS! Phase 2 Data Pipeline is 100% operational. Proceed to Phase 3.
